# 24B The MCP tool server
The catalog is closed, every tool has an action class, and scope is checked by the server on every call.

In [1]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

fixture mode: True | clock: 2026-10-15T06:00:00-04:00


In [2]:
cat = pd.DataFrame([(t, c, k or "(session)", catalog.policy_for(t)) for t, (c, k) in catalog.CATALOG.items()],
                   columns=["tool", "action class", "scope", "policy"])
print(cat.to_string(index=False))

                tool   action class     scope    policy
       list_mandates           READ (session)      auto
        bdc_holdings           READ    public      auto
         bdc_signals           READ    public      auto
          get_filing           READ (session)      auto
     mandate_profile           READ   mandate      auto
      watchlist_read           READ   mandate      auto
         memory_read           READ   mandate      auto
memory_propose_write INTERNAL_WRITE   mandate    notify
     watchlist_write INTERNAL_WRITE   mandate   approve
         draft_email          DRAFT   mandate      auto
    request_approval        CONTROL (session) principal
          send_email  EXTERNAL_SEND   mandate   approve


In [3]:
# Your server must declare exactly this catalog. A conformance check on a server with three mistakes:
mine = {t: c for t, (c, _) in catalog.CATALOG.items()}
mine.pop("watchlist_read"); mine["export_all_mandates"] = "READ"; mine["send_email"] = "INTERNAL_WRITE"
for p in catalog.conformance(mine):
    print("  ", p)

   missing tool: watchlist_read
   extra tool (the catalog is closed): export_all_mandates
   send_email: class INTERNAL_WRITE, expected EXTERNAL_SEND


In [4]:
s = channel.login("priya", "priya-fixture-token", "mandate_a", USERS)
led = audit.Ledger(clock=clock.clock)
tools = standin.read_tools(s, led, PANEL, MANDATES, M)        # four READ tools over the fixture
print("public signals:", len(tools["bdc_signals"](scope="public")))
print("own mandate:", tools["mandate_profile"](scope="mandate_a")["client"])
for call in [lambda: tools["mandate_profile"](scope="mandate_b"), lambda: tools["mandate_profile"](scope="public"),
             lambda: channel.require_host("https://example.com/exfil?x=1", led)]:
    try:
        call()
    except (channel.ScopeError, PermissionError) as e:
        print("refused:", type(e).__name__, e)
print(pd.DataFrame(led.find("policy_block"))[["event_id", "kind", "rule_id", "scope", "host"]].fillna(""))

public signals: 6
own mandate: Harbor State Employees' Retirement System
refused: ScopeError scope 'mandate_b' not allowed
refused: ScopeError scope 'public' not allowed
refused: PermissionError host 'example.com' not allowed
   event_id          kind  rule_id      scope         host
0  ev-00003  policy_block  R-SCOPE  mandate_b             
1  ev-00004  policy_block  R-SCOPE     public             
2  ev-00005  policy_block   R-HOST             example.com


## Wrapping it in MCP (SDK v2)
```python
from mcp.server import MCPServer            # pip install "mcp[cli]>=2.1,<3"
mcp = MCPServer("creditwatch-tools")

@mcp.tool()
def bdc_signals(scope: str) -> list[dict]:
    """Within-filing credit signals with evidence (READ)."""
    return tools["bdc_signals"](scope=scope)  # scope checked inside, every call
```
Type hints become the input schema and the docstring becomes the description. Run it with `mcp dev services/mcp/server.py` to open the Inspector. The cell below runs only if `mcp` is installed.

In [5]:
import importlib.util, asyncio
if importlib.util.find_spec("mcp") is None:
    print("mcp not installed: skipped (pip install 'mcp[cli]>=2.1,<3')")
else:
    try:
        from mcp.server import MCPServer
        from mcp import Client
        srv = MCPServer("creditwatch-demo")
        @srv.tool()
        def bdc_signals(scope: str) -> list[dict]:
            """Within-filing credit signals with evidence (READ)."""
            return tools["bdc_signals"](scope=scope)
        async def demo():
            async with Client(srv) as c:                  # in memory: no transport
                return await c.call_tool("bdc_signals", {"scope": "public"})
        print(asyncio.run(demo()))
    except Exception as e:
        print("MCP demo failed:", type(e).__name__, e, "(check the SDK version against the v2 docs)")

MCP demo failed: RuntimeError asyncio.run() cannot be called from a running event loop (check the SDK version against the v2 docs)


/tmp/ipykernel_29103/2270989345.py:18: RuntimeWarning: coroutine 'demo' was never awaited
  print("MCP demo failed:", type(e).__name__, e, "(check the SDK version against the v2 docs)")


**Your turn.** Build `services/mcp/` with exactly the Appendix C catalog, Pydantic schemas, typed errors and a contract test per tool (Part 1). `catalog.conformance` must return `[]` for your server's registry.

### Part 1: the server (`services/mcp/`)
- `errors.py`: typed errors (`BadInput`, `NotFound`, `ScopeDenied`, `PolicyBlocked`, `ApprovalInvalid`), each with a code and a decision.
- `schemas.py`: a Pydantic input model (extra fields forbidden) and output model per tool.
- `tools.py`: `Toolbox`, bound to one session; every call is schema, then scope, then tool, then one audit event and one action record.
- `server.py`: `REGISTRY` and the MCP wrappers. `mcp dev services/mcp/server.py` opens the Inspector.

Contract tests: `pytest -q tests/test_mcp.py`.

In [6]:
sys.path.insert(0, str(ROOT))
import tempfile
from services.chat.session import load_world
from services.mcp.server import REGISTRY
from services.mcp.tools import Toolbox
from services.mcp.errors import ToolFailure
print("conformance:", catalog.conformance(REGISTRY), "(must be [])")
tmp = pathlib.Path(tempfile.mkdtemp())
tb = Toolbox(s, load_world(ROOT), workspace.fresh_workspace(ROOT / P["agent"]["seed"], tmp / "agent"), tmp / "outputs",
             audit.Ledger(clock=clock.clock))
d = tb.call("draft_email", scope="mandate_a", to=["cio@harborstate.example"], subject="Juliett", body="Marks are down.")
tb.call("request_approval", item_id=d.draft_id)
tok = tb.approve(d.draft_id)                                  # Priya approves this exact payload (harness, not a tool)
print("sent:", tb.call("send_email", scope="mandate_a", draft_id=d.draft_id, approval_token=tok))
for tool, kw in [("mandate_profile", {"scope": "mandate_b"}),
                 ("draft_email", {"scope": "mandate_a", "to": ["cio@harborstate.example"], "subject": "s", "body": "Northgate holds Hotel."}),
                 ("send_email", {"scope": "mandate_a", "draft_id": d.draft_id, "approval_token": "forged"})]:
    try:
        tb.call(tool, **kw)
    except ToolFailure as e:
        print("refused:", tool, e.as_dict())
print(pd.DataFrame(tb.actions))

conformance: [] (must be [])
sent: outbox_message_id='mock-0001' status='sent_to_mock_outbox'
refused: mandate_profile {'error': 'SCOPE_DENIED', 'rule_id': 'R-SCOPE', 'detail': "scope 'mandate_b' not allowed"}
refused: draft_email {'error': 'POLICY_BLOCKED', 'rule_id': 'R-OTHER-MANDATE', 'detail': 'standing rule R-OTHER-MANDATE fired'}
refused: send_email {'error': 'APPROVAL_INVALID', 'rule_id': 'R-APPROVAL', 'detail': 'approval token malformed'}
               tool   action_class     args_hash   decision
0       draft_email          DRAFT  3e55cfd25261   executed
1  request_approval        CONTROL  a5823bff2ebe  requested
2        send_email  EXTERNAL_SEND  13b58ff383cb   executed
3   mandate_profile           READ  3c42b7eacba8    blocked
4       draft_email          DRAFT  44970ad20f13    blocked
5        send_email  EXTERNAL_SEND  773a2141936e    blocked


In [7]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests" / "test_mcp.py")], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-800:] or r.stderr[-800:])
assert r.returncode == 0

................                                                         [100%]
16 passed in 13.74s

